In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
import json

try:
    nombre_carpeta = dbutils.widgets.get("folder_name")
    cod_indicador = dbutils.widgets.get("indicador_code")
except Exception:
    # Valores por defecto si no están definidos
    #nombre_carpeta = "pbi"
    #cod_indicador = "NY.GDP.MKTP.CD"
    print(f"✅ Widgets listos. Procesando: {nombre_carpeta}")


class BronzeIngestor:
    def __init__(self, indicador_nombre, esquema):
        self.indicador = indicador_nombre
        self.esquema = esquema
        self.tabla_destino = f"socioeconomics.bronze.bronze_{self.indicador}"
        

    def ejecutar_ingesta_batch(self, ruta_archivo_especifico):
        print(f"📦 Iniciando refresco total (Overwrite) desde: {ruta_archivo_especifico}")
    
        # Lectura Batch
        df_batch = (spark.read
            .option("multiLine", "true")
            .schema(self.esquema)
            .json(ruta_archivo_especifico))

        # Agregamos metadatos de trazabilidad
        df_final = df_batch.withColumn("_ingestado_el", F.current_timestamp()) \
                           .withColumn("_archivo_origen", F.lit(ruta_archivo_especifico))

        # Escritura Delta con Overwrite
        df_final.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(self.tabla_destino)
    
        print(f"✅ Tabla {self.tabla_destino} sobrescrita con éxito.")

In [0]:
#%run ./00_extractor_landing.ipynb
#%run ./schema_world_bank.ipynb

extractor = SocioDataExtractor(nombre_carpeta)
ruta_del_archivo_nuevo = extractor.obtener_datos(cod_indicador)

# 2. Ingesta (Solo si la extracción fue exitosa)
if ruta_del_archivo_nuevo:
    ingestor = BronzeIngestor(nombre_carpeta, SCHEMA_WORLD_BANK)
    ingestor.ejecutar_ingesta_batch(ruta_del_archivo_nuevo)